# Student Pass/Fail Prediction using Classification Algorithms

## Objective

The objective of this project is to predict whether a student will pass or fail based on academic and study-related factors using multiple classification algorithms and compare their performance.

## 1. Import Required Libraries

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    f1_score
)

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

The Student Performance Factors dataset was loaded for classification analysis.

In [13]:
df = pd.read_csv("../data/student_performance_dataset.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 3. Dataset Preview

The first five records were examined to understand the structure of the dataset.

In [14]:
df.head()

,Student_ID,Gender,Study_Hours_per_Week,Attendance_Rate,Past_Exam_Scores,Parental_Education_Level,Internet_Access_at_Home,Extracurricular_Activities,Final_Exam_Score,Pass_Fail
0,S147,Male,31,68.267841,86,High School,Yes,Yes,63,Pass
1,S136,Male,16,78.222927,73,PhD,No,No,50,Fail
2,S209,Female,21,87.525096,74,PhD,Yes,No,55,Fail
3,S458,Female,27,92.076483,99,Bachelors,No,No,65,Pass
4,S078,Female,37,98.655517,63,Masters,No,Yes,70,Pass


## 4. Dataset Information

The dataset information was examined to identify data types and missing values.

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  708 non-null    str    
 1   Gender                      708 non-null    str    
 2   Study_Hours_per_Week        708 non-null    int64  
 3   Attendance_Rate             708 non-null    float64
 4   Past_Exam_Scores            708 non-null    int64  
 5   Parental_Education_Level    708 non-null    str    
 6   Internet_Access_at_Home     708 non-null    str    
 7   Extracurricular_Activities  708 non-null    str    
 8   Final_Exam_Score            708 non-null    int64  
 9   Pass_Fail                   708 non-null    str    
dtypes: float64(1), int64(3), str(6)
memory usage: 55.4 KB


## 5. Dataset Dimensions

The number of rows and columns in the dataset were identified.

In [16]:
rows, columns = df.shape

print(f"Rows    : {rows}")
print(f"Columns : {columns}")

Rows    : 708
Columns : 10


## 6. Statistical Summary

Basic statistical measures were calculated for numerical features.

In [17]:
df.describe()

,Study_Hours_per_Week,Attendance_Rate,Past_Exam_Scores,Final_Exam_Score
count,708.000000,708.000000,708.000000,708.000000
mean,26.132768,78.107722,77.871469,58.771186
std,8.877727,13.802802,14.402739,6.705877
min,10.000000,50.116970,50.000000,50.000000
25%,19.000000,67.550094,65.000000,52.000000
50%,27.000000,79.363046,79.000000,59.500000
75%,34.000000,89.504232,91.000000,64.000000
max,39.000000,99.967675,100.000000,77.000000


## 7. Missing Value Analysis

The dataset was checked for missing values before model training.

In [18]:
df.isnull().sum()

Student_ID                    0
Gender                        0
Study_Hours_per_Week          0
Attendance_Rate               0
Past_Exam_Scores              0
Parental_Education_Level      0
Internet_Access_at_Home       0
Extracurricular_Activities    0
Final_Exam_Score              0
Pass_Fail                     0
dtype: int64

### Observation

No missing values were found in the dataset. Therefore, no data cleaning was required before model training.

## 8. Encode Target Variable

The target variable was converted into numerical values for machine learning.

In [19]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()

df["Pass_Fail"] = target_encoder.fit_transform(df["Pass_Fail"])

df["Pass_Fail"].head()

0    1
1    0
2    0
3    1
4    1
Name: Pass_Fail, dtype: int64

### Observation

The `Pass_Fail` column was encoded using Label Encoding.

- **Pass = 1**
- **Fail = 0**

In [22]:
df.groupby("Pass_Fail")["Final_Exam_Score"].describe()

,count,mean,std,min,25%,50%,75%,max
Pass_Fail,,,,,,,,
0,354.0,52.980226,3.216838,50.0,50.0,52.0,56.0,59.0
1,354.0,64.562147,3.529372,60.0,62.0,64.0,67.0,77.0


## 10. Remove Unnecessary Features

The `Student_ID` column was removed because it is a unique identifier and does not contribute to prediction.

The `Final_Exam_Score` column was also removed to prevent **data leakage**, since the target variable (`Pass_Fail`) is directly derived from the final exam score.

In [23]:
df.drop(columns=["Student_ID", "Final_Exam_Score"], inplace=True)

df.head()

,Gender,Study_Hours_per_Week,Attendance_Rate,Past_Exam_Scores,Parental_Education_Level,Internet_Access_at_Home,Extracurricular_Activities,Pass_Fail
0,Male,31,68.267841,86,High School,Yes,Yes,1
1,Male,16,78.222927,73,PhD,No,No,0
2,Female,21,87.525096,74,PhD,Yes,No,0
3,Female,27,92.076483,99,Bachelors,No,No,1
4,Female,37,98.655517,63,Masters,No,Yes,1


### Observation

The `Student_ID` and `Final_Exam_Score` columns were removed before model training. This ensures that the model learns from student-related factors rather than directly relying on the exam score used to determine the target variable.

## 11. Encode Categorical Features

Categorical features were converted into numerical values using Label Encoding to make them suitable for machine learning algorithms.

In [24]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

categorical_columns = [
    "Gender",
    "Parental_Education_Level",
    "Internet_Access_at_Home",
    "Extracurricular_Activities"
]

for column in categorical_columns:
    df[column] = encoder.fit_transform(df[column])

df.head()

,Gender,Study_Hours_per_Week,Attendance_Rate,Past_Exam_Scores,Parental_Education_Level,Internet_Access_at_Home,Extracurricular_Activities,Pass_Fail
0,1,31,68.267841,86,1,1,1,1
1,1,16,78.222927,73,3,0,0,0
2,0,21,87.525096,74,3,1,0,0
3,0,27,92.076483,99,0,0,0,1
4,0,37,98.655517,63,2,0,1,1


### Observation

The categorical variables were converted into numerical values using **Label Encoding**.

| Feature | Encoding |
|----------|----------|
| Gender | Female = 0, Male = 1 |
| Parental_Education_Level | Bachelors = 0, High School = 1, Masters = 2, PhD = 3 |
| Internet_Access_at_Home | No = 0, Yes = 1 |
| Extracurricular_Activities | No = 0, Yes = 1 |
| Pass_Fail | Fail = 0, Pass = 1 |

All categorical features are now in numerical format and are ready for machine learning models.

## 12. Feature Selection

The input features (`X`) and target variable (`y`) were separated for model training.

In [28]:
X = df.drop(columns=["Pass_Fail"])

y = df["Pass_Fail"]

print("Feature Matrix Shape :", X.shape)
print("Target Vector Shape  :", y.shape)

Feature Matrix Shape : (708, 7)
Target Vector Shape  : (708,)


## 13. Train-Test Split

The dataset was divided into training and testing sets using an 80:20 ratio.

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)

print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

Training Features : (566, 7)
Testing Features  : (142, 7)
Training Labels   : (566,)
Testing Labels    : (142,)


## 14. Feature Scaling

Feature Scaling was applied using **StandardScaler** to normalize numerical features before training the models.

Scaling is especially important for distance-based algorithms like K-Nearest Neighbors (KNN).

In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully.")

Feature scaling completed successfully.


### Observation

The training data was fitted using the StandardScaler, and the same transformation was applied to the test data. This prevents data leakage and ensures consistent feature scaling.

## 15. Logistic Regression

A Logistic Regression classifier was trained as the baseline classification model.

In [31]:
logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_scaled, y_train)

y_pred_logistic = logistic_model.predict(X_test_scaled)

### Logistic Regression Performance

In [32]:
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)

f1_logistic = f1_score(y_test, y_pred_logistic)

print(f"Accuracy : {accuracy_logistic:.4f}")
print(f"F1 Score : {f1_logistic:.4f}")

Accuracy : 0.7817
F1 Score : 0.7862


### Observation

The Logistic Regression model achieved:

- **Accuracy:** 78.17%
- **F1 Score:** 78.62%

The model provided a good baseline performance and was able to classify pass and fail students with reasonable accuracy.

In [33]:
print(classification_report(y_test, y_pred_logistic))

              precision    recall  f1-score   support

           0       0.79      0.76      0.78        71
           1       0.77      0.80      0.79        71

    accuracy                           0.78       142
   macro avg       0.78      0.78      0.78       142
weighted avg       0.78      0.78      0.78       142



### Observation

The Logistic Regression model achieved an overall **Accuracy of 78.17%**.

- **Fail (Class 0):** Precision = 79%, Recall = 76%, F1 Score = 78%
- **Pass (Class 1):** Precision = 77%, Recall = 80%, F1 Score = 79%

Both classes contain an equal number of samples (71 each), resulting in similar Macro and Weighted average scores. The model demonstrates balanced performance in predicting both Pass and Fail students.